In [25]:
import os
import cv2
import numpy as np
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tkinter import Tk, Label, Button, filedialog
from PIL import Image, ImageTk
from keras.models import load_model

# Define paths
train_dir = r"C:\Users\Pritam\Desktop\Major Project\Major Project\Train_Augmented-20250124T062019Z-001\Train_dir"
validation_dir = r"C:\Users\Pritam\Desktop\Major Project\Major Project\Train_Augmented-20250124T062019Z-001\Val_dir"
test_dir = r"C:\Users\Pritam\Desktop\Major Project\Major Project\Train_Augmented-20250124T062019Z-001\Test_dir"
model_path = r"C:\Users\Pritam\Desktop\Major Project\Major Project\Train_Augmented-20250124T062019Z-001"

In [26]:
# Constants
image_size = (128, 128)  # Resize all images to this size
num_classes = 38  # Number of subclasses

# Create and compile the DCNN model
def create_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [27]:
# Create the model
model = create_model()

# Display the summary table
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_12 (Conv2D)          (None, 126, 126, 32)      896       
                                                                 
 max_pooling2d_12 (MaxPooli  (None, 63, 63, 32)        0         
 ng2D)                                                           
                                                                 
 conv2d_13 (Conv2D)          (None, 61, 61, 64)        18496     
                                                                 
 max_pooling2d_13 (MaxPooli  (None, 30, 30, 64)        0         
 ng2D)                                                           
                                                                 
 conv2d_14 (Conv2D)          (None, 28, 28, 128)       73856     
                                                                 
 max_pooling2d_14 (MaxPooli  (None, 14, 14, 128)      

In [28]:
# Train the model
def train_model():
    datagen = ImageDataGenerator(rescale=1.0/255, horizontal_flip=True, rotation_range=20, zoom_range=0.2)

    train_generator = datagen.flow_from_directory(
        train_dir, target_size=image_size, batch_size=32, class_mode='categorical')

    validation_generator = datagen.flow_from_directory(
        validation_dir, target_size=image_size, batch_size=32, class_mode='categorical')

    model = create_model()

    model.fit(
        train_generator,
        epochs=50,
        validation_data=validation_generator
    )

    model.save(model_path)
    print("Model saved as",model_path)

In [29]:
train_model()

Found 41199 images belonging to 38 classes.
Found 5150 images belonging to 38 classes.
Epoch 1/50
1288/1288 [==============================] - 667s 516ms/step - loss: 2.5078 - accuracy: 0.2801 - val_loss: 1.4520 - val_accuracy: 0.5627
Epoch 2/50
1288/1288 [==============================] - 360s 279ms/step - loss: 1.6156 - accuracy: 0.5058 - val_loss: 0.8754 - val_accuracy: 0.7483
Epoch 3/50
1288/1288 [==============================] - 359s 279ms/step - loss: 1.2821 - accuracy: 0.5978 - val_loss: 0.7034 - val_accuracy: 0.7961
Epoch 4/50
1288/1288 [==============================] - 358s 278ms/step - loss: 1.0734 - accuracy: 0.6578 - val_loss: 0.5325 - val_accuracy: 0.8311
Epoch 5/50
1288/1288 [==============================] - 359s 279ms/step - loss: 0.9253 - accuracy: 0.7065 - val_loss: 0.4750 - val_accuracy: 0.8441
Epoch 6/50
1288/1288 [==============================] - 595s 462ms/step - loss: 0.8089 - accuracy: 0.7380 - val_loss: 0.4258 - val_accuracy: 0.8553
Epoch 7/50
1288/1288 [===

INFO:tensorflow:Assets written to: C:\Users\Pritam\Desktop\Major Project\Major Project\Train_Augmented-20250124T062019Z-001\assets


Model saved as C:\Users\Pritam\Desktop\Major Project\Major Project\Train_Augmented-20250124T062019Z-001


In [42]:
# Predict a single image
def predict_image(model, image_path):
    image = load_img(image_path, target_size=image_size)
    image = img_to_array(image) / 255.0
    image = np.expand_dims(image, axis=0)
    prediction = model.predict(image)
    class_idx = np.argmax(prediction)
    return class_idx

In [43]:
# Interface for user interaction
def user_interface():
    def upload_image():
        file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
        if file_path:
            img = Image.open(file_path)
            img.thumbnail((300, 300))
            img = ImageTk.PhotoImage(img)
            image_label.config(image=img)
            image_label.image = img

            # Predict
            class_idx = predict_image(loaded_model, file_path)
            result_label.config(text=f"Predicted Class: {class_idx}")

    def capture_image():
        cap = cv2.VideoCapture(0)
        while True:
            ret, frame = cap.read()
            cv2.imshow("Capture Image", frame)
            if cv2.waitKey(1) & 0xFF == ord('c'):
                img_path = 'captured_image.jpg'
                cv2.imwrite(img_path, frame)
                cap.release()
                cv2.destroyAllWindows()
                
                # Display and predict
                img = Image.open(img_path)
                img.thumbnail((300, 300))
                img = ImageTk.PhotoImage(img)
                image_label.config(image=img)
                image_label.image = img

                class_idx = predict_image(loaded_model, img_path)
                result_label.config(text=f"Predicted Class: {class_idx}")
                break

    # Load the trained model
    global loaded_model
    loaded_model = load_model(model_path)

    # Create the UI
    root = Tk()
    root.title("Plant Leaf Disease Classification")

    upload_button = Button(root, text="Upload Image", command=upload_image)
    upload_button.pack()

    capture_button = Button(root, text="Capture Image", command=capture_image)
    capture_button.pack()
    
    image_label = Label(root)
    image_label.pack()

    result_label = Label(root, text="Predicted Class: None")
    result_label.pack()

    root.mainloop()

In [44]:
user_interface()

1/1 [==============================] - 0s 93ms/step
